# Clasificador de texto generado por IA v2 (español, tesis de educación)

Fine-tuning de un RoBERTa en español sobre segmentos de prosa de 150-250 palabras
(`dataset_v2.zip`, exportado con `python manage.py exportar_dataset_ia_v2`).

**Qué cambia respecto al v1:** el v1 solo vio humanos UPeU ≤ 2021 y marcaba como IA a humanos
recientes (Wikipedia, UPeU 2022-26). El v2 suma **~340 artículos de Wikipedia en español** (educación,
pedagogía, psicología, ciencias sociales, historia y ciencia; 40 reservados que **nunca** entran al
entrenamiento) y mantiene los UPeU ≤ 2021. La IA es la misma del v1. Sigue siendo **indicativo**.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

Grupos de humanos (todos separados por documento):
- `train`: UPeU ≤ 2021 + Wikipedia de entrenamiento.
- **Wikipedia reservada** (`test_wiki`) y **UPeU test** (`humano_corpus`, partición `test`): FP medido y
  umbral calibrado aquí.
- Solo diagnóstico, independientes del umbral: 20 tesis UPeU ≤ 2021 de control, 20 tesis UPeU 2022-26
  (`humano_reciente`) y 6 artículos de Wikipedia del v1 (`wikipedia_v1`).

Experimentos:
1. **Dejando un LLM fuera:** entrenar con Gemini-API y probar en Claude, y al revés.
2. **Modelo final:** Gemini-API + Claude; se prueba en los textos de chat (ChatGPT = LLM nunca visto).

**Umbral:** el más bajo con el que Wikipedia reservada **y** UPeU test quedan, cada una, con FP ≤ 5 %
(el mayor de los dos cuantiles). Como se calibra en esos mismos grupos, su FP sale ≤ 5 % por
construcción: lo que decide es el recall y los FP de los grupos de diagnóstico, que no intervienen.

**Criterio para desplegar:** FP ≤ 5 % en Wikipedia reservada y en UPeU test **y** recall ≥ 80 % en el
LLM dejado fuera (mínimo de los dos cruces Gemini ↔ Claude).

In [ ]:
!nvidia-smi -L
!pip -q install -U "transformers>=4.44" "accelerate>=0.33" scikit-learn

## 1. Subir el dataset

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.exists("segmentos.jsonl"):
    subido = files.upload()  # elige dataset_v2.zip
    nombre = next(iter(subido))
    zipfile.ZipFile(nombre).extractall(".")
print(open("resumen.json", encoding="utf-8").read()[:1800])

In [ ]:
import pandas as pd
df = pd.read_json("segmentos.jsonl", lines=True)
print(len(df), "segmentos")
print(df.groupby(["origen", "particion", "solape"]).size().to_string())
# Ningún documento puede estar en dos particiones.
assert (df.groupby("documento")["particion"].nunique() == 1).all()
# Wikipedia reservada jamás entra al entrenamiento.
assert not set(df[df.particion == "test_wiki"]["documento"]) & set(df[df.particion == "train"]["documento"])
print("humanos:", int((df.etiqueta == 0).sum()), "(tope 3500)")

## 2. Modelo y entrenamiento

In [ ]:
import json, time, numpy as np, torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)

MAX_LENGTH = 384
EPOCHS = 3
SEED = 42
CANDIDATOS = ["PlanTL-GOB-ES/roberta-base-bne", "dccuchile/bert-base-spanish-wwm-cased"]

MODELO_BASE = None
for nombre in CANDIDATOS:
    try:
        tokenizer = AutoTokenizer.from_pretrained(nombre)
        AutoModelForSequenceClassification.from_pretrained(nombre, num_labels=2)
        MODELO_BASE = nombre
        break
    except Exception as exc:
        print("No carga", nombre, "->", exc)
print("Modelo base:", MODELO_BASE)

class Segmentos(Dataset):
    def __init__(self, frame):
        self.enc = tokenizer(frame["texto"].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = frame["etiqueta"].tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

class TrainerConPesos(Trainer):
    """Pérdida con pesos por clase (hay muchos más humanos que IA)."""
    def __init__(self, *a, pesos=None, **kw):
        super().__init__(*a, **kw)
        self.pesos = pesos
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=self.pesos.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def entrenar(train_df, salida):
    set_seed(SEED)
    modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=2)
    n = len(train_df); n_ia = int(train_df["etiqueta"].sum()); n_h = n - n_ia
    pesos = torch.tensor([n / (2 * n_h), n / (2 * n_ia)], dtype=torch.float)
    print(f"train: {n_h} humanos, {n_ia} IA, pesos {pesos.tolist()}")
    args = TrainingArguments(
        output_dir=salida, num_train_epochs=EPOCHS, learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        warmup_steps=max(1, int(0.1 * EPOCHS * -(-n // 16))), weight_decay=0.01, fp16=torch.cuda.is_available(),
        save_strategy="no", logging_steps=50, report_to=[], seed=SEED)
    trainer = TrainerConPesos(model=modelo, args=args, train_dataset=Segmentos(train_df),
                              data_collator=DataCollatorWithPadding(tokenizer), pesos=pesos)
    t0 = time.time(); trainer.train(); print(f"entrenado en {(time.time()-t0)/60:.1f} min")
    return trainer

def probabilidades(trainer, frame):
    if len(frame) == 0:
        return np.array([])
    pred = trainer.predict(Segmentos(frame))
    return torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()

## 3. Grupos y métricas

In [ ]:
from sklearn.metrics import roc_auc_score

def cuantil_fp(prob_humanos, fp_max=0.05):
    """Menor umbral con el que, como mucho, el 5 % de esos humanos queda por encima."""
    return float(np.quantile(prob_humanos, 1 - fp_max, method="higher")) + 1e-6

def umbral_v2(p_wiki, p_upeu):
    """FP <= 5 % en Wikipedia reservada Y en UPeU test: el mayor de los dos cuantiles."""
    return max(cuantil_fp(p_wiki), cuantil_fp(p_upeu))

def pct(p, umbral):
    return round(100 * float((p >= umbral).mean()), 2) if len(p) else None

base = lambda f: f[~f["solape"]]
sel = lambda o, p: df[(df["origen"] == o) & (df["particion"] == p)]
H_TRAIN = df[(df["etiqueta"] == 0) & (df["particion"] == "train")]
WIKI_RES = sel("humano_wikipedia", "test_wiki")
UPEU_TEST = sel("humano_corpus", "test")
DIAG = {  # solo diagnóstico: no intervienen en el umbral
    "control UPeU<=2021": df[df["origen"] == "humano_control"],
    "UPeU 2022-26": df[df["origen"] == "humano_reciente"],
    "Wikipedia del v1": df[df["origen"] == "wikipedia_v1"],
}
API = {o: df[df["origen"] == f"{o}_api"] for o in ["gemini", "claude"]}
CHAT = df[df["particion"] == "test_chat"]
print(f"wiki reservada: {len(WIKI_RES)} segm / {WIKI_RES.documento.nunique()} docs; "
      f"UPeU test: {len(UPEU_TEST)} segm / {UPEU_TEST.documento.nunique()} docs")
for k, v in DIAG.items():
    print(f"{k}: {len(v)} segm / {v.documento.nunique()} docs")

def evaluar(nombre, trainer, fuera=None, estilos=None):
    """FP por grupo de humanos con el umbral v2 y, si hay, recall del LLM `fuera`."""
    p_wiki = probabilidades(trainer, WIKI_RES)
    p_upeu = probabilidades(trainer, UPEU_TEST)
    umbral = umbral_v2(p_wiki, p_upeu)
    fila = {"experimento": nombre, "umbral": round(umbral, 4),
            "fp_wikipedia_reservada_%": pct(p_wiki, umbral), "fp_upeu_test_%": pct(p_upeu, umbral)}
    for k, frame in DIAG.items():
        fila[f"fp_{k}_%"] = pct(probabilidades(trainer, frame), umbral)
    if fuera is not None:
        p_ia = probabilidades(trainer, fuera)
        p_h = np.r_[p_wiki, p_upeu]
        fila["auroc"] = round(roc_auc_score(np.r_[np.zeros(len(p_h)), np.ones(len(p_ia))],
                                            np.r_[p_h, p_ia]), 4)
        fila["recall_ia_%"] = pct(p_ia, umbral)
        if estilos is not None:
            for estilo in ["normal", "estudiante", "humanizado"]:
                m = np.asarray(estilos) == estilo
                if m.any():
                    fila[f"recall_{estilo}_%"] = pct(p_ia[m], umbral)
    return fila, umbral

RESULTADOS = []

## 4. Dejando un LLM fuera (Gemini-API ↔ Claude)

In [ ]:
for entrena, prueba in [("gemini", "claude"), ("claude", "gemini")]:
    print(f"=== entrenar con {entrena}, probar en {prueba} ===")
    fuente = API[entrena]
    train_df = pd.concat([H_TRAIN, fuente[fuente["particion"] == "train"]])
    trainer = entrenar(train_df, f"out_{entrena}")
    fuera = base(API[prueba])  # TODOS sus documentos, sin ventanas solapadas
    fila, _ = evaluar(f"{entrena}->{prueba}", trainer, fuera, fuera["estilo"])
    RESULTADOS.append(fila)
    del trainer; torch.cuda.empty_cache()
print(pd.DataFrame(RESULTADOS).T.to_string())

## 5. Modelo final (Gemini-API + Claude)

Se entrena con los humanos de train (UPeU ≤ 2021 + Wikipedia) y **todos** los documentos de IA por API.
Wikipedia reservada y UPeU test quedan fuera: sin ellos no habría forma honesta de medir FP.

In [ ]:
ia_api = pd.concat([API["gemini"], API["claude"]])
train_final = pd.concat([H_TRAIN, ia_api[(ia_api["particion"] == "train") | ~ia_api["solape"]]])
final = entrenar(train_final, "out_final")

p_wiki = probabilidades(final, WIKI_RES)
p_upeu = probabilidades(final, UPEU_TEST)
UMBRAL = umbral_v2(p_wiki, p_upeu)
p_chat = probabilidades(final, CHAT)
p_diag = {k: probabilidades(final, v) for k, v in DIAG.items()}

fila_final, _ = evaluar("final->chat (60 + 6 docx)", final, CHAT, CHAT["estilo"])
RESULTADOS.append(fila_final)
for origen in ["chatgpt_chat", "gemini_chat", "ia_docx"]:
    m = (CHAT["origen"] == origen).to_numpy()
    fila, _ = evaluar(f"final->{origen}", final, CHAT[m], CHAT["estilo"][m])
    RESULTADOS.append(fila)
tabla = pd.DataFrame(RESULTADOS)
print(tabla.T.to_string())

In [ ]:
# Criterio para desplegar.
fuera = tabla[tabla["experimento"].isin(["gemini->claude", "claude->gemini"])]
f = tabla[tabla["experimento"] == "final->chat (60 + 6 docx)"].iloc[0]
chatgpt = tabla[tabla["experimento"] == "final->chatgpt_chat"].iloc[0]
criterio = {
    "fp_wikipedia_reservada_%": f["fp_wikipedia_reservada_%"],
    "fp_upeu_test_%": f["fp_upeu_test_%"],
    "recall_llm_fuera_min_%": float(fuera["recall_ia_%"].min()),
    # Informativos (no deciden): independientes del umbral y LLM nunca visto.
    "fp_control_upeu_%": f["fp_control UPeU<=2021_%"],
    "fp_upeu_2022_26_%": f["fp_UPeU 2022-26_%"],
    "fp_wikipedia_v1_%": f["fp_Wikipedia del v1_%"],
    "recall_chatgpt_nunca_visto_%": chatgpt["recall_ia_%"],
    "recall_chat_final_%": f["recall_ia_%"],
}
criterio["desplegar"] = bool(
    criterio["fp_wikipedia_reservada_%"] <= 5
    and criterio["fp_upeu_test_%"] <= 5
    and criterio["recall_llm_fuera_min_%"] >= 80
)
print(json.dumps(criterio, indent=1))

# % de segmentos sobre el umbral por documento de chat (como lo verá el sistema).
chat = CHAT.assign(prob=p_chat, ia=p_chat >= UMBRAL)
print(chat.groupby(["origen", "documento"])["ia"].mean().mul(100).round(1).groupby("origen").describe().to_string())
# Idem para los documentos humanos de diagnóstico (el 21-70 % del v1).
for k, v in DIAG.items():
    d = v.assign(ia=p_diag[k] >= UMBRAL).groupby("documento")["ia"].mean().mul(100).round(1)
    print(f"\n{k}: % de segmentos marcados IA por documento ->", d.describe().round(1).to_dict())

## 6. Guardar y descargar el modelo final (.zip)

In [ ]:
import shutil
destino = "clasificador_ia_v2"
final.save_model(destino)
tokenizer.save_pretrained(destino)
with open(f"{destino}/umbral.json", "w", encoding="utf-8") as fh:
    json.dump({"version": "v2", "modelo_base": MODELO_BASE, "max_length": MAX_LENGTH, "epocas": EPOCHS,
               "umbral": UMBRAL, "fp_objetivo": 0.05, "criterio": criterio,
               "resultados": RESULTADOS}, fh, ensure_ascii=False, indent=1, default=float)
pd.DataFrame({"documento": CHAT["documento"], "origen": CHAT["origen"], "estilo": CHAT["estilo"],
              "prob": p_chat}).to_csv(f"{destino}/predicciones_chat.csv", index=False)
pd.DataFrame({"documento": WIKI_RES["documento"], "prob": p_wiki}).to_csv(
    f"{destino}/predicciones_wikipedia_reservada.csv", index=False)
pd.DataFrame({"documento": UPEU_TEST["documento"], "prob": p_upeu}).to_csv(
    f"{destino}/predicciones_upeu_test.csv", index=False)
shutil.make_archive("clasificador_ia_v2", "zip", destino)
files.download("clasificador_ia_v2.zip")